# Notebook 05: Parameter-Efficient Fine-Tuning (PEFT) with LoRA

In this notebook, we use **LoRA (Low-Rank Adaptation)** to fine-tune the Moirai encoder.
Instead of updating millions of parameters, LoRA freezes the foundation model and injects tiny trainable rank-decomposition matrices into the linear layers.

uv pip install peft

In [5]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
from IPython.display import display


from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import *
from models import *
from utils import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid=[1e-4]
wd_grid=[0.05]

BATCH_SIZE = 256

In [6]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [7]:
METRICS_COLS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted Precision", "Weighted Recall", "Weighted F1"]
df_patch_8_metrics = pd.DataFrame(columns=METRICS_COLS)
df_patch_8_metrics.index.name = "Model"

## 2. LoRA Baseline: Mean Pooling on Full Sequence (Context + Mask)

In [8]:
methods = ["LoRA", "DoRA", "AdaLoRA"]
df_mean_pool = pd.DataFrame(index=methods, columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"
df_mean_pool.index.name = "PEFT Method"

for method in methods:
    print(f"\n{'='*40}\nTraining with {method}\n{'='*40}")
    for patch in PATCH_SIZES:
        use_dora = (method == "DoRA")
        use_adalora = (method == "AdaLoRA")

        _, metrics = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": MeanPoolingClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": 8,
                "use_dora": use_dora,
                "use_adalora": use_adalora
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_mean_pool.loc[method, patch] = metrics["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"MeanPool-{method}"] = metrics

print("Mean Pooling (LoRA/DoRA/AdaLoRA) - Accuracy")
display(df_mean_pool.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))


Training with LoRA
LR=0.0001| WD=0.05
 Early stopping : 36
Val Loss: 1.0932
LR=0.0001| WD=0.05
 Early stopping : 38
Val Loss: 1.0900
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.0871
LR=0.0001| WD=0.05
 Early stopping : 31
Val Loss: 1.1504

Training with DoRA
LR=0.0001| WD=0.05
 Early stopping : 41
Val Loss: 1.0704
LR=0.0001| WD=0.05
 Early stopping : 33
Val Loss: 1.0689
LR=0.0001| WD=0.05
 Early stopping : 38
Val Loss: 1.0917
LR=0.0001| WD=0.05
 Early stopping : 31
Val Loss: 1.1275

Training with AdaLoRA
LR=0.0001| WD=0.05
 Early stopping : 110
Val Loss: 1.0290
LR=0.0001| WD=0.05
 Early stopping : 82
Val Loss: 1.0422
LR=0.0001| WD=0.05
 Early stopping : 87
Val Loss: 1.0611
LR=0.0001| WD=0.05
 Early stopping : 75
Val Loss: 1.0970
Mean Pooling (LoRA/DoRA/AdaLoRA) - Accuracy


Patch Size,8,16,32,64
PEFT Method,,,,
LoRA,0.6436,0.6285,0.6338,0.6253
DoRA,0.6488,0.6277,0.6314,0.6346
AdaLoRA,0.6403,0.6440,0.6419,0.6318



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
MeanPool-LoRA,0.6436,0.5026,0.3789,0.3851,0.5975,0.6436,0.5905
MeanPool-DoRA,0.6488,0.4401,0.3893,0.3976,0.5896,0.6488,0.6033
MeanPool-AdaLoRA,0.6403,0.4374,0.3922,0.3997,0.5879,0.6403,0.6025


## 3. LoRA Rank Ablation Study
Testing the impact of the bottleneck rank $r$ on the MHA performance. A higher rank means more trainable parameters.

In [ ]:
PATCH = 8
RANKS_TO_TEST = [2, 4, 8, 16, 32, 64]
methods = ["LoRA", "DoRA", "AdaLoRA"]

df_rank_ablation8 = pd.DataFrame(index=methods, columns=RANKS_TO_TEST)
df_rank_ablation8.index.name = "PEFT Method"
df_rank_ablation8.columns.name = f"Rank 'r' (Patch {PATCH})"

for method in methods:
    for r in RANKS_TO_TEST:
        use_dora = (method == "DoRA")
        use_adalora = (method == "AdaLoRA")

        _, metrics = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": MeanPoolingClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": r,
                "use_dora": use_dora,
                "use_adalora": use_adalora
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            lr_grid=lr_grid, wd_grid=wd_grid
        )
        df_rank_ablation8.loc[method, r] = metrics["Accuracy"]
        df_patch_8_metrics.loc[f"{method} r={r}"] = metrics

print(f"Rank Ablation (Patch = {PATCH}) - Accuracy")
display(df_rank_ablation8.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

LR=0.0001| WD=0.05
 Early stopping : 54
Val Loss: 1.0520
LR=0.0001| WD=0.05
 Early stopping : 44
Val Loss: 1.0809
LR=0.0001| WD=0.05
 Early stopping : 41
Val Loss: 1.0879
LR=0.0001| WD=0.05
 Early stopping : 34
Val Loss: 1.0588
LR=0.0001| WD=0.05
 Early stopping : 31
Val Loss: 1.0559
LR=0.0001| WD=0.05
 Early stopping : 22
Val Loss: 1.0690
LR=0.0001| WD=0.05
 Early stopping : 56
Val Loss: 1.0553
LR=0.0001| WD=0.05
 Early stopping : 46
Val Loss: 1.0733
LR=0.0001| WD=0.05
 Early stopping : 37
Val Loss: 1.0871
LR=0.0001| WD=0.05
 Early stopping : 26
Val Loss: 1.0953
LR=0.0001| WD=0.05
 Early stopping : 28
Val Loss: 1.0537
LR=0.0001| WD=0.05
 Early stopping : 20
Val Loss: 1.0932
LR=0.0001| WD=0.05


## 4. Advanced LoRA: Multi-Head Attention (Context + Mask)

In [ ]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = [f"MHA (Heads={NUM_HEADS} | LoRA r=64)"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_mha = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE,
                "remove_last_patch": False, "lora_r": 64
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = metrics_mha["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"MHA-16 ({mode}) LoRA r=64"] = metrics_mha

In [ ]:
print("MHA with LoRA - Accuracy")
display(df_adv_single.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))